# Phase picking using PhaseNet (INSTANCE pre-trained model)

This notebook applies the PhaseNet model pre-trained on the INSTANCE dataset to detect P-wave and S-wave arrivals in the accelerometric signals.

## 1. Imports and visualization settings

In [ ]:
import seisbench
import seisbench.models as sbm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging
import sys
import pickle
from pathlib import Path
from src  import (
    apply_phasenet_to_signals,
    set_plot_style,
    add_hypocentral_distance,
    expand_to_component_level,
    add_coda_onsets_to_dataframe,
    segment_all_signals,
    quality_control_all_stations,
    print_quality_control_summary,
    add_time_columns,
    convert_signals_to_dict,
    validate_signals_dict,
    merge_phasenet_picks_with_metadata,
    get_station_from_filename
)
colors, colors1 = set_plot_style()
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()
def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)
logger.info("Environment ready")

## 2. Configuration

In [ ]:
# CONFIGURATION
DATA_TYPE = 'acceleration'  # Options: 'acceleration', 'velocity', 'displacement'

# Determine signal column name and units based on DATA_TYPE
if DATA_TYPE == 'acceleration':
    SIGNAL_COLUMN = 'acceleration'
    SIGNAL_UNIT = 'cm/s²'
    PEAK_COLUMN = 'PGA_CM/S^2'
    TIME_PEAK_COLUMN = 'TIME_PGA_S'
elif DATA_TYPE == 'velocity':
    SIGNAL_COLUMN = 'velocity'
    SIGNAL_UNIT = 'cm/s'
    PEAK_COLUMN = 'PGV_CM/S'
    TIME_PEAK_COLUMN = 'TIME_PGV_S'
elif DATA_TYPE == 'displacement':
    SIGNAL_COLUMN = 'displacement'
    SIGNAL_UNIT = 'cm'
    PEAK_COLUMN = 'PGD_CM'
    TIME_PEAK_COLUMN = 'TIME_PGD_S'
else:
    raise ValueError(f"Unknown DATA_TYPE: {DATA_TYPE}")

logger.info(f"Working with {DATA_TYPE} data")
logger.info(f"Signal column: {SIGNAL_COLUMN}")
logger.info(f"Peak column: {PEAK_COLUMN}")

## 3. Data loading

In [ ]:
# Get project root
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Define all paths from project root with DATA_TYPE subdirectories
METADATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed' / '01a_metadata' / DATA_TYPE
SIGNALS_PROCESSED_IMPORT = PROJECT_ROOT / 'data' / 'processed' / '01b_signals' / DATA_TYPE
SIGNALS_PROCESSED_EXPORT = PROJECT_ROOT / 'data' / 'processed' / '03_phase_identification_phasenet' / DATA_TYPE
FIGURES_DIR = PROJECT_ROOT / 'figures' / '03b_phase_identification_phasenet' / DATA_TYPE
LATEX_TABLES_DIR = PROJECT_ROOT / 'data' / 'processed' / 'latex_tables' / DATA_TYPE

# Create output directories
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METADATA_PROCESSED.mkdir(parents=True, exist_ok=True)
SIGNALS_PROCESSED_IMPORT.mkdir(parents=True, exist_ok=True)
SIGNALS_PROCESSED_EXPORT.mkdir(parents=True, exist_ok=True)
LATEX_TABLES_DIR.mkdir(parents=True, exist_ok=True)

check(FIGURES_DIR.exists(), f"Figures directory ready: {FIGURES_DIR}")
check(SIGNALS_PROCESSED_EXPORT.exists(), f"Exported signals directory ready: {SIGNALS_PROCESSED_EXPORT}")
check(LATEX_TABLES_DIR.exists(), f"LaTeX tables directory ready: {LATEX_TABLES_DIR}")
check(METADATA_PROCESSED.exists(), f"Processed metadata directory ready: {METADATA_PROCESSED}")
check(SIGNALS_PROCESSED_IMPORT.exists(), f"Processed signals directory ready: {SIGNALS_PROCESSED_IMPORT}")

# Load metadata
logger.info(f"Loading {DATA_TYPE} metadata...")
df_meta = pd.read_parquet(METADATA_PROCESSED / f'metadata_clean_{DATA_TYPE[:3]}.parquet')
check(df_meta is not None, "Metadata loaded successfully")
check(len(df_meta) > 0, "Metadata dataframe is not empty")
logger.info(f"Metadata loaded, shape: {df_meta.shape}")

# Load signals
logger.info(f"Loading {DATA_TYPE} signals...")
df_signals = pd.read_parquet(SIGNALS_PROCESSED_IMPORT / f'{DATA_TYPE[:3]}_preprocessed_scaling.parquet')
check(df_signals is not None, "Signals loaded successfully")
check(len(df_signals) > 0, "Signals dataframe is not empty")
logger.info(f"Signals loaded, shape: {df_signals.shape}")
logger.info(f"Unique files: {df_signals['file'].nunique()}")

## 4. Metadata preparation

In [ ]:
logger.info("Preparing station metadata (1 row per station)...")

# Select essential columns and reduce to 1 row per station
df_meta_stations = df_meta.drop_duplicates('STATION_CODE')[[
    'STATION_CODE',
    'STATION_LATITUDE_DEGREE',
    'STATION_LONGITUDE_DEGREE',
    'EPICENTRAL_DISTANCE_KM',
    'INSTRUMENTAL_FREQUENCY_HZ',
    'LOW_CUT_FREQUENCY_HZ',
    'HIGH_CUT_FREQUENCY_HZ',
    PEAK_COLUMN,
    TIME_PEAK_COLUMN,
    'EVENT_DATE',
    'EVENT_DEPTH_KM',
    'DATE_TIME_FIRST_SAMPLE'
]].copy()

logger.info("Calculating hypocentral distance for QC...")
hypo_depth_km = df_meta_stations['EVENT_DEPTH_KM'].iloc[0]
df_meta_stations = add_hypocentral_distance(
    df_meta_stations,
    hypo_depth_km=hypo_depth_km,
    distance_col='EPICENTRAL_DISTANCE_KM'
)

check('hypocentral_distance_km' in df_meta_stations.columns, "Hypocentral distance calculated")

n_stations = len(df_meta_stations)
logger.info(f"Station metadata ready: {n_stations} unique stations")

# Display first rows
print("\nFirst 5 stations:")
display(df_meta_stations.head())

# Summary statistics
print("\nEpicentral distance range:")
print(f"  Min: {df_meta_stations['EPICENTRAL_DISTANCE_KM'].min():.2f} km")
print(f"  Max: {df_meta_stations['EPICENTRAL_DISTANCE_KM'].max():.2f} km")
print(f"  Median: {df_meta_stations['EPICENTRAL_DISTANCE_KM'].median():.2f} km")

## 5. Load PhaseNet model pre-trained on INSTANCE

In [ ]:
logger.info("Loading PhaseNet model pre-trained on INSTANCE...")
model = sbm.PhaseNet.from_pretrained("instance")
check(model is not None, "PhaseNet model loaded successfully")

## 6. Phase picking with resampling to 100 Hz

In [ ]:
# Phase picking with PhaseNet
logger.info(f"Starting PhaseNet inference on {DATA_TYPE} signals...")

df_picks = apply_phasenet_to_signals(
    df_signals=df_signals,
    model=model,
    signal_column=SIGNAL_COLUMN,
    sampling_rate_original=200,
    sampling_rate_target=100,
    min_p_probability=0.1,
    min_s_probability=0.1
)

logger.info(f"PhaseNet inference complete: {len(df_picks)} stations processed")

# Quality check and filtering
logger.info("Checking PhaseNet picks quality...")

# Check for invalid picks (S before or at P)
invalid_picks = df_picks[df_picks['t_s_seconds'] <= df_picks['t_p_seconds']]
if len(invalid_picks) > 0:
    logger.warning(f"{len(invalid_picks)} stations with S ≤ P")
    logger.warning(str(invalid_picks[['station_code', 't_p_seconds', 't_s_seconds']]))

# Filter out stations with missing onsets (NaN)
n_before = len(df_picks)
df_picks = df_picks[
    df_picks['t_p_seconds'].notna() & 
    df_picks['t_s_seconds'].notna()
].copy()
n_after = len(df_picks)

if n_before > n_after:
    logger.warning(f"Filtered out {n_before - n_after} station(s) with missing onsets")
else:
    logger.info("All picks valid (no missing onsets)")

# Extract valid station list
valid_stations = df_picks['station_code'].unique().tolist()

# Summary
logger.info(f"\nPhaseNet picks summary:")
logger.info(f"  Valid stations: {len(valid_stations)}")
logger.info(f"  Mean P probability: {df_picks['p_probability_max'].mean():.3f}")
logger.info(f"  Mean S probability: {df_picks['s_probability_max'].mean():.3f}")
logger.info(f"  Mean S-P time: {(df_picks['t_s_seconds'] - df_picks['t_p_seconds']).mean():.2f}s")

# Save cleaned results
output_path = SIGNALS_PROCESSED_EXPORT / f'phasenet_picks_{DATA_TYPE}.csv'
df_picks.to_csv(output_path, index=False)
logger.info(f"\nSaved: {output_path}")

## 7. Merge picks with metadata

In [ ]:
logger.info("Merging PhaseNet picks with station metadata...")

df_meta_stations = merge_phasenet_picks_with_metadata(df_picks, df_meta_stations)

check(len(df_meta_stations) > 0, f"Metadata merged: {len(df_meta_stations)} stations")
check('origin_time_seconds' in df_meta_stations.columns, "origin_time calculated from metadata")

logger.info(f"Origin time range: {df_meta_stations['origin_time_seconds'].min():.2f}s - {df_meta_stations['origin_time_seconds'].max():.2f}s")

## 8. Convert signals to dictionary

In [ ]:
logger.info("Adding time column to signals...")
df_signals = add_time_columns(
    df_signals, 
    df_meta, 
    time_col='DATE_TIME_FIRST_SAMPLE',
    sampling_interval_col='SAMPLING_INTERVAL_S'
)
check('time' in df_signals.columns, "Time column added to signals")

# Filter signals to only include valid stations
logger.info(f"Filtering signals to valid stations ({len(valid_stations)} stations)...")

df_signals = df_signals[
    df_signals['file'].apply(get_station_from_filename).isin(valid_stations)
].copy()

logger.info(f"Filtered signals: {len(df_signals)} rows (from {len(df_signals)} total)")

# Convert DataFrame to dict
signals_dict = convert_signals_to_dict(df_signals,
                                       signal_column=SIGNAL_COLUMN)

check(len(signals_dict) > 0, "Signals dictionary created")
logger.info(f"Dictionary contains {len(signals_dict)} stations")

# Validate structure
print("\nValidating signals dictionary...")
report = validate_signals_dict(signals_dict)

check(report['valid'], "All signals validated successfully")

## Expand to component level

In [ ]:
logger.info("Expanding metadata to component level...")

# Filter df_meta to only include valid stations
df_meta_valid = df_meta[df_meta['STATION_CODE'].isin(valid_stations)].copy()
logger.info(f"Filtered df_meta: {len(df_meta_valid)} components ({len(df_meta)} total)")

# Expand using filtered metadata
df_full = expand_to_component_level(df_meta_stations, df_meta_valid)

check(len(df_full) == 63, f"Expanded to component level: {len(df_full)} components")
logger.info(f"Component-level DataFrame shape: {df_full.shape}")

## 9. Coda onset detection

In [ ]:
logger.info("Calculating coda onsets for all methods...")

# Check for missing onsets
n_total = len(df_full)
n_missing_p = df_full['t_p_detected_seconds'].isna().sum()
n_missing_s = df_full['t_s_detected_seconds'].isna().sum()

if n_missing_p > 0 or n_missing_s > 0:
    logger.warning(f"Missing onsets: P={n_missing_p}, S={n_missing_s}")
    logger.warning("Filtering out components with missing onsets...")
    
    df_full_valid = df_full[
        df_full['t_p_detected_seconds'].notna() & 
        df_full['t_s_detected_seconds'].notna()
    ].copy()
    
    logger.info(f"Valid components: {len(df_full_valid)}/{n_total}")
else:
    df_full_valid = df_full.copy()

# Coda detection
df_full = add_coda_onsets_to_dataframe(
    df_full_valid, 
    signals_dict,
    threshold_arias=0.95,
    threshold_envelope=0.3,
    sampling_rate=200,
    unit='samples'
)

check('t_coda_rautian_seconds' in df_full.columns, "Coda onsets computed")
logger.info("Coda onsets computed for all methods: rautian, arias, envelope, median")

# Summary
print("\nCoda onset summary (mean S-wave duration):")
for method in ['rautian', 'arias', 'envelope', 'median']:
    col = f's_duration_{method}_seconds'
    if col in df_full.columns:
        mean_dur = df_full[col].mean()
        std_dur = df_full[col].std()
        print(f"  {method.capitalize():10s}: {mean_dur:.2f} ± {std_dur:.2f}s")

## 10. Windowing

In [ ]:
for method in ['rautian', 'arias', 'envelope', 'median']:
    logger.info(f"Segmenting signals with {method} coda method...")
    
    # Windowing
    windowed_signals = segment_all_signals(
        signals_dict, 
        df_full, 
        coda_method=method
    )
    
    # Save
    output_file = SIGNALS_PROCESSED_EXPORT / f'windowed_{DATA_TYPE}_{method}_phasenet.pkl'
    with open(output_file, 'wb') as f:
        pickle.dump(windowed_signals, f)
    
    logger.info(f"Saved: {output_file}")
    logger.info(f"File size: {output_file.stat().st_size / 1e6:.2f} MB")

## Quality control

In [ ]:
logger.info("Running quality control on windowed signals...")
logger.info("=" * 80)

# Run QC per ogni metodo di coda
qc_results = {}

for method in ['rautian', 'arias', 'envelope', 'median']:
    logger.info(f"\n{method.upper()} METHOD")
    logger.info("-" * 40)
    
    # Load windowed signals
    input_file = SIGNALS_PROCESSED_EXPORT / f'windowed_{DATA_TYPE}_{method}_phasenet.pkl'
    
    if not input_file.exists():
        logger.warning(f"File not found: {input_file}")
        continue
    
    with open(input_file, 'rb') as f:
        windowed_signals = pickle.load(f)
    
    # Run QC
    qc_result = quality_control_all_stations(
        windowed_signals,
        df_full,
        df_meta_stations,
        peak_column=PEAK_COLUMN,
        time_peak_column=TIME_PEAK_COLUMN,
        snr_threshold=3.0,
        coda_method=method
    )
    
    qc_results[method] = qc_result
    
    # Print summary
    print_quality_control_summary(qc_result)

logger.info("\n" + "=" * 80)
logger.info("Quality control complete for all methods")